# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Note on Section 1:** I couldn't find a document literally named "the FlyRank research paper" in the public repo (not in `docs/`, and `submission/paper_url.txt` is still the placeholder). I used the bundled reference `outputs/model_report.md` instead, since it's the closest thing in-repo to a findings writeup. **If your portal card links a different paper, swap the two findings below for that one** — the audit *method* stays identical.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — "`random_forest` is the best model, Precision@50 = 0.740, chosen via a `client_holdout` split."**
- **Label source:** `is_declining_label = (trend_direction == "down")`, and `trend_direction` compares `impressions_last_30d` vs `impressions_prev_30d` (days 31-60 back) — a >20% drop = "down."
- **Does the validation design carry the claim?** The `client_holdout` split is the right instinct — it stops the model from memorizing a client's quirks. But it doesn't check a different failure: several of the model's `_90d` features (impressions, clicks, sessions, `days_with_impressions`...) are aggregated over a 90-day window that **structurally contains** the label's own last-30-day window. A client-grouped split can't catch that, because the leak isn't between clients — it's inside every single row. Precision@50=0.740 could be partly real skill and partly the model reading 30 of its 90 input-days from the same days that decide the answer.
- **Constructive next step:** before trusting 0.740, run the confession test from the leakage skill — refit with the last-30-day contribution subtracted out of every `_90d` feature (or restricted to days 61-90 back only) and see how far the number falls.

**Finding 2 — "Top feature: `days_with_impressions` (importance 0.1578), ahead of `log_impressions_90d` (0.1282) and `avg_position` (0.1090)."**
- **Label source:** same as above.
- **Does the validation design carry the claim?** Same gap. `days_with_impressions` counts, out of the full 90-day window, how many days had at least one impression — and the label's exact last-30-day window is part of that count. A page whose last 30 days went quiet (the thing `trend_direction` is measuring) will *by construction* show up with a lower 90-day day-count too, some of the time for a reason unrelated to any real signal the model learned. This is close to the textbook "future/overlapping window" case, not a circumstantial resemblance.
- **Constructive next step:** this is exactly the kind of top feature the leakage skill says to be suspicious of, not celebrate — worth checking whether the model's ranking power collapses without it before it's called the top signal.

I ran the actual with/without check on my own model in Section 3 below, since I have that feature set on hand — the spread does shrink, though not all the way to nothing.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quoting the numbers from outputs/model_report.md for reference (no re-computation needed here --
# that report belongs to the reference pipeline, not my own model).
print("Finding 1: random_forest, Precision@50=0.740, split=client_holdout")
print("Finding 2: top feature days_with_impressions, importance=0.1578")
print("Label: is_declining_label = (trend_direction == 'down'); trend_direction compares last_30d vs prev_30d impressions")


Finding 1: random_forest, Precision@50=0.740, split=client_holdout
Finding 2: top feature days_with_impressions, importance=0.1578
Label: is_declining_label = (trend_direction == 'down'); trend_direction compares last_30d vs prev_30d impressions


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My ML-08 K-Means archetype model was fit and evaluated **in-sample** — clustered on all 30,000 rows, then ranked and scored on those same rows. That's the "before" number. For "after," I split the 32 pseudonymous clients 70/30 (22 train, 10 test), fit the scaler + K-Means on train rows only, then **assigned** test rows to the nearest already-fitted cluster and used the **train-derived** decline rate per cluster (never touching test labels) to build the priority order — only then scored precision@K on the held-out client rows.

| | Base rate | Precision@20 | Precision@50 |
|---|---|---|---|
| **Before** (in-sample, fit+eval on all 30,000 rows) | 0.542 | 0.50 | 0.46 |
| **After** (client-grouped: fit on 22 clients, eval on 10 held-out clients) | 0.502 | **0.35** | **0.32** |

The honest number is noticeably worse, and now sits *below* even the held-out base rate at both cuts. That lines up with the ARI=0.59 reseed instability I already found in ML-08 — the archetypes don't reform identically on a different slice of clients, so a priority order learned from training clusters doesn't transfer cleanly to a fresh cluster fit on unseen clients. I can't explain this gap away; it's the honest finding, not a bug to patch.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

def build_feat(d):
    f = pd.DataFrame(index=d.index)
    f["log_impressions"] = np.log1p(d["impressions_90d"])
    f["ctr"] = d["ctr"].fillna(0)
    pos = d["avg_position"].replace(0, np.nan)
    f["avg_position"] = pos.fillna(pos.median())
    f["engagement_rate"] = d["engagement_rate"].fillna(0)
    f["word_count"] = d["word_count"].fillna(d["word_count"].median())
    f["content_age_days"] = d["content_age_days"]
    f["days_since_last_update"] = d["days_since_last_update"]
    return f

feat = build_feat(df)
k = 6

clients = df["client_id"].unique().tolist()
rng = np.random.RandomState(42)
rng.shuffle(clients)
n_train = int(len(clients) * 0.7)
train_clients, test_clients = set(clients[:n_train]), set(clients[n_train:])
is_train = df["client_id"].isin(train_clients).values
is_test = ~is_train
print(f"train clients={len(train_clients)} ({is_train.sum()} rows)  test clients={len(test_clients)} ({is_test.sum()} rows)")

scaler = StandardScaler().fit(feat[is_train])
X_train, X_test = scaler.transform(feat[is_train]), scaler.transform(feat[is_test])

km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
df_train = df[is_train].assign(cluster=km.labels_)
df_test = df[is_test].assign(cluster=km.predict(X_test))

priority_order = df_train.groupby("cluster")["is_declining_label"].mean().sort_values(ascending=False).index.tolist()
df_test["priority_tier"] = df_test["cluster"].map({c: i for i, c in enumerate(priority_order)})
test_queue = df_test.sort_values(["priority_tier", "impressions_90d"], ascending=[True, False])

print(f"\nAFTER (honest, client-grouped): base_rate={df_test['is_declining_label'].mean():.3f}  "
      f"P@20={test_queue['is_declining_label'].head(20).mean():.3f}  "
      f"P@50={test_queue['is_declining_label'].head(50).mean():.3f}")
print("BEFORE (in-sample, from ML-08): base_rate=0.542  P@20=0.500  P@50=0.460")


train clients=22 (21763 rows)  test clients=10 (8237 rows)

AFTER (honest, client-grouped): base_rate=0.502  P@20=0.350  P@50=0.320
BEFORE (in-sample, from ML-08): base_rate=0.542  P@20=0.500  P@50=0.460


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the ML-06 attack checklist against my ML-08 feature set: `log_impressions, ctr, avg_position, engagement_rate, word_count, content_age_days, days_since_last_update`.

- **Label-derived / sibling columns in features?** No — `trend_direction`, `trend_pct`, `is_declining_label` are all excluded.
- **Product flags / existing-system scores as features?** No — the starter CSV ships no `health_score` or prior action-score column, and none is used.
- **Future/overlapping windows?** **Yes, partially** — `log_impressions`, `ctr`, `avg_position`, and `engagement_rate` are all built from the 90-day GSC/GA4 window, which contains the label's own last-30-day window (same issue flagged against the reference paper in Section 1). `word_count`, `content_age_days`, and `days_since_last_update` don't touch that window at all and are clean.
- **Confession test (with vs. without the 4 overlapping features):** decline-rate spread across the 6 clusters drops from **0.505** (with) to **0.364** (without) — a real drop, so the overlapping features are doing some genuine work in separating archetypes. But it's not a collapse to near-zero either: word count and staleness alone still carry more than two-thirds of the separation. **My honest read: some of ML-08's archetype separation leans on window-overlapping features I didn't check for until now — not disqualifying, but worth disclosing rather than treating the clustering as fully independent of the label's time window.**
- **Population selection:** all 30,000 rows used unconditionally, nothing filtered on outcome-window information.
- **Split grouped by repeating entity?** Done in Section 2 (`client_id`).
- **Base rate printed next to every metric?** Yes, throughout.
- **"Too good to be true" check:** silhouette scores topped out at 0.30 (k=6) — modest, not suspiciously perfect, which is mild reassurance but doesn't substitute for the window check above.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
overlapping = ["log_impressions", "ctr", "avg_position", "engagement_rate"]
safe = ["word_count", "content_age_days", "days_since_last_update"]

print("Label-derived cols in feature set:", set(["trend_direction", "trend_pct", "is_declining_label"]) & set(feat.columns))
print("Overlapping-window features:", overlapping)
print("Window-clean features:", safe)

for name, cols in [("WITH overlapping features (full ML-08 set)", overlapping + safe),
                   ("WITHOUT overlapping features (safe-only)", safe)]:
    X = StandardScaler().fit_transform(feat[cols])
    km_c = KMeans(n_clusters=6, random_state=42, n_init=10).fit(X)
    d2 = df.assign(cluster=km_c.labels_)
    rates = d2.groupby("cluster")["is_declining_label"].mean().sort_values(ascending=False)
    spread = rates.max() - rates.min()
    print(f"\n{name}: decline-rate spread={spread:.3f} (max={rates.max():.3f}, min={rates.min():.3f})")


Label-derived cols in feature set: set()
Overlapping-window features: ['log_impressions', 'ctr', 'avg_position', 'engagement_rate']
Window-clean features: ['word_count', 'content_age_days', 'days_since_last_update']

WITH overlapping features (full ML-08 set): decline-rate spread=0.505 (max=0.647, min=0.143)

WITHOUT overlapping features (safe-only): decline-rate spread=0.364 (max=0.641, min=0.277)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from ML-08, Section 4):** *"That's not a bug to fix with a better k; it's a ceiling this method has."*

That's an unqualified, general claim — "a ceiling this method has" reads as true of K-Means clustering for this task everywhere, forever, which my one dataset and one feature set can't support. It also predates this notebook's own finding that part of the little separation clustering *did* achieve was leaning on window-overlapping features — so the original sentence was confident about a limitation without yet knowing one input to that limitation was itself questionable.

**Rewrite:** *"In this 30,000-row anonymized slice, the K-Means archetype queue did not beat the rule baseline at precision@20 or precision@50, even under in-sample evaluation, and performance measured, honestly, under a client-grouped split (Section 2). Part of the modest separation it did achieve appears associated with features that overlap the label's own time window (Section 3), and part survives without them. This is directional evidence that static, mostly-visibility-and-staleness features aren't enough for this queue to beat a hand rule on this data — not a general finding about clustering as a method, and not yet tested at warehouse scale or with trend-safe features."*

Slower to read, but every clause is something I actually measured, and it stops short of claims about clustering "as a method" that nothing here tested.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No further computation needed -- this section rewrites language, backed by the numbers in sections 2 and 3 above.
print("See sections 2-3 for the evidence this rewrite is scoped to.")


See sections 2-3 for the evidence this rewrite is scoped to.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.